# Seminar 2. Custom PyTorch Operators

# Семинар 2. Пользовательские операторы PyTorch

# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular  
- Reusable  
- Readable  
- Easy to extend  


## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks

---

# Построение моделей в PyTorch через композицию

Модели в PyTorch строятся с использованием **композиции**.  
Вместо того чтобы определять одну большую монолитную сеть, мы создаём модель, объединяя более мелкие и переиспользуемые модули.

Каждый модуль может содержать другие модули, что позволяет строить иерархичные и хорошо структурированные архитектуры.

---

## Композиция

Композиция означает:

- Модель строится из более маленьких блоков.
- Каждый блок может содержать несколько слоёв.
- Блоки можно повторно использовать в более крупных архитектурах.
- Сложные модели создаются путём объединения более простых компонентов.

Это делает код:

- Модульным  
- Переиспользуемым  
- Читаемым  
- Легко расширяемым  


## Ключевые идеи

- Наследоваться от `nn.Module`
- Определять слои внутри `__init__`
- Описывать вычисления в `forward()`
- Создавать переиспользуемые блоки
- Строить более крупные модели, комбинируя блоки

---

## Пример: модель из двух блоков

Ниже приведён простой пример, где:

- Мы определяем переиспользуемые блоки: `LinearReLUBlock` и `LinearTanhBlock`
- Итоговая модель состоит из двух таких блоков


In [3]:
import torch
import torch.nn as nn


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

CombinedModel(
  (block1): LinearReLUBlock(
    (linear): Linear(in_features=4, out_features=8, bias=True)
    (activation): ReLU()
  )
  (block2): LinearTanhBlock(
    (linear): Linear(in_features=8, out_features=8, bias=True)
    (activation): Tanh()
  )
  (output): Linear(in_features=8, out_features=2, bias=True)
)


# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us

---

# Что даёт `nn.Module`

Когда мы наследуемся от `nn.Module`, мы автоматически получаем мощную функциональность, которая работает **рекурсивно для всех подмодулей**.

## Что предоставляет `nn.Module`



### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.

---

### Регистрация параметров

Все слои, назначенные как атрибуты (например, `self.linear = nn.Linear(...)`):

- Автоматически регистрируются
- Попадают в `model.parameters()`
- Включаются в `model.state_dict()`

Это работает **рекурсивно** для всех вложенных блоков.

In [4]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)


Registered parameters:
block1.linear.weight torch.Size([8, 4])
block1.linear.bias torch.Size([8])
block2.linear.weight torch.Size([8, 8])
block2.linear.bias torch.Size([8])
output.weight torch.Size([2, 8])
output.bias torch.Size([2])


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.

---

### Автоматическое отслеживание градиентов

Во время прямого прохода:

- PyTorch динамически строит граф вычислений
- Вызов `loss.backward()` вычисляет градиенты
- Градиенты сохраняются в `.grad` каждого параметра

Ручное управление графом не требуется.

In [5]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:",
      model.output.weight.grad is not None)
model.output.weight.grad


Gradient computed for output layer: True


tensor([[ 0.0344,  0.1324, -0.0375,  0.0997, -0.0076,  0.0879,  0.0575,  0.0797],
        [-0.2521, -0.3974,  0.1033, -0.2816, -0.0068, -0.3524, -0.0968, -0.1282]])

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.

---

### Перенос на устройство и смена типа (`.to()`)

Вызов:

    model.to(device)

или

    model.to(dtype)

автоматически переносит:

- Параметры
- Буферы
- Подмодули

на CPU/GPU и/или в нужный `dtype`.

In [6]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

Model device: cpu
Model dtype: torch.float32


### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.

---

### Сохранение и загрузка (`state_dict()`)

- `model.state_dict()` рекурсивно возвращает все параметры
- `model.load_state_dict(...)` восстанавливает их

Это работает для всего дерева модулей.

In [7]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type        | Behavior in `train()` Mode                  |
|------------------|--------------------------------------------|
| `Dropout`         | Randomly zeroes some activations           |
| `BatchNorm`       | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type        | Behavior in `eval()` Mode                   |
|------------------|--------------------------------------------|
| `Dropout`         | Passes all activations through unchanged  |
| `BatchNorm`       | Uses stored running mean/variance         |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.

---

# Режимы `train()` и `eval()` в PyTorch

У модулей PyTorch есть два основных режима: **режим обучения** и **режим оценки (инференса)**.  
Переключение между ними влияет на слои, поведение которых отличается при обучении и при использовании модели.

---

## `model.train()`

- Переводит модель в **режим обучения**.
- Используется при обучении модели с обновлением параметров.
- Влияет на некоторые слои, например:

| Тип слоя          | Поведение в режиме `train()`                |
|-------------------|---------------------------------------------|
| `Dropout`         | Случайно зануляет часть активаций           |
| `BatchNorm`       | Обновляет скользящую статистику (среднее/дисперсия) |

- Градиенты вычисляются как обычно.

---

## `model.eval()`

- Переводит модель в **режим оценки (инференса)**.
- Используется при валидации, тестировании и деплое.
- Влияет на некоторые слои:

| Тип слоя          | Поведение в режиме `eval()`                 |
|-------------------|---------------------------------------------|
| `Dropout`         | Пропускает активации без изменений          |
| `BatchNorm`       | Использует сохранённые средние/дисперсии    |

- Слои не обновляют внутреннюю статистику.
- Градиенты обычно не нужны (часто используется вместе с `torch.no_grad()`).

---

## Ключевые моменты

- Во время обучения всегда используйте `model.train()`.
- Во время оценки или тестирования всегда используйте `model.eval()`.
- Если забыть переключить режим, результаты могут быть нестабильными, особенно при `Dropout` и `BatchNorm`.



In [8]:
import torch
import torch.nn as nn

# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)


Training mode output:
 tensor([[ 0.0063, -0.1080],
        [ 0.4254, -0.0025],
        [-0.2246, -0.7912],
        [-0.9286, -0.4939],
        [ 0.5067,  0.3356]], grad_fn=<AddmmBackward0>)
Evaluation mode output:
 tensor([[ 0.2109,  0.0162],
        [ 0.2011,  0.1220],
        [ 0.0306, -0.0866],
        [-0.0845, -0.1623],
        [ 0.5532,  0.4275]])


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.

---

# `torch.no_grad()` и `torch.inference_mode()` в PyTorch

При инференсе (когда мы используем модель без обновления параметров) в PyTorch есть контекстные менеджеры для **отключения отслеживания градиентов**. Это экономит память и ускоряет вычисления.

---

## `torch.no_grad()`

- Отключает отслеживание градиентов.
- Полезен во время оценки и инференса.
- Градиенты **не вычисляются**, но autograd всё ещё отслеживает часть операций для внутренних целей.
- Можно использовать как **контекстный менеджер** или **декоратор функции**.

---

## `torch.inference_mode()`

- Добавлен в PyTorch 1.9.
- Похож на `no_grad()`, но **более эффективен**.
- Полностью отключает autograd и снижает потребление памяти.
- Рекомендуется для чистых inference-пайплайнов.



In [10]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 2)

    # -----------------------------
    # Using torch.no_grad() as method decorator
    # -----------------------------

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    @torch.no_grad()
    def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.inference_mode() as method decorator
    # -----------------------------
    @torch.inference_mode()
    def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


model = SimpleModel()
x = torch.randn(5, 4)

# -----------------------------
# Call decorated methods
# -----------------------------
output_no_grad_method = model.forward_no_grad(x)
output_infer_method = model.forward_inference(x)

print("Output no_grad method:\n", output_no_grad_method)
print("Output inference_mode method:\n", output_infer_method)

# -----------------------------
# Using context managers
# -----------------------------

with torch.no_grad():
    output_no_grad_cm = model(x)

with torch.inference_mode():
    output_infer_cm = model(x)

print("Output no_grad context manager:\n", output_no_grad_cm)
print("Output inference_mode context manager:\n", output_infer_cm)


Output no_grad method:
 tensor([[-0.0269,  0.2931],
        [-0.4911,  0.4358],
        [ 0.4754, -0.4218],
        [-0.0566,  0.2830],
        [-0.3495, -0.0282]])
Output inference_mode method:
 tensor([[-0.0269,  0.2931],
        [-0.4911,  0.4358],
        [ 0.4754, -0.4218],
        [-0.0566,  0.2830],
        [-0.3495, -0.0282]])
Output no_grad context manager:
 tensor([[-0.0269,  0.2931],
        [-0.4911,  0.4358],
        [ 0.4754, -0.4218],
        [-0.0566,  0.2830],
        [-0.3495, -0.0282]])
Output inference_mode context manager:
 tensor([[-0.0269,  0.2931],
        [-0.4911,  0.4358],
        [ 0.4754, -0.4218],
        [-0.0566,  0.2830],
        [-0.3495, -0.0282]])


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.

---

# Отключение градиентов с помощью `requires_grad_(False)`

В PyTorch есть удобный метод `requires_grad_()`, который позволяет **включать или отключать градиенты на месте (in-place)** для параметров модели или тензора.

Использование:

```python
param.requires_grad_(False)
```

- Устанавливает `requires_grad=False` **in-place** для данного параметра.
- Это полезно для «заморозки» модели при инференсе или transfer learning.
- Можно применить ко всей модели, рекурсивно проходя по её параметрам.


In [11]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

fc.weight: requires_grad=False
fc.bias: requires_grad=False
Output shape: torch.Size([5, 2])


# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.  

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact

---

# Переопределение `train()` и `eval()` в `nn.Module`

`nn.Module` в PyTorch предоставляет встроенные методы `train(mode: bool = True)` и `eval()` для переключения между **режимом обучения** и **режимом оценки**.  

Иногда при создании **кастомных модулей или блоков** полезно **переопределить эти методы**, чтобы выполнять дополнительные действия при смене режима.

---

## Зачем переопределять?

- Применять логику, зависящую от режима, к нестандартным подблокам или атрибутам
- Логировать и отслеживать переключения режимов
- Автоматически менять внутренние флаги или буферы вместе с режимом `train/eval`

---

## Как это работает

- `train(mode: bool = True)` устанавливает `self.training = mode` для модуля
- `eval()` эквивалентен `train(False)`
- Реализация по умолчанию рекурсивно вызывает `train(mode)` для всех подмодулей
- Переопределение позволяет добавить кастомное поведение, сохраняя рекурсивные обновления


In [12]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Self

class CustomBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x

    # -----------------------------
    # Override train() method
    # -----------------------------
    def train(self, mode: bool = True) -> Self:
        print(f"CustomBlock set to {'train' if mode else 'eval'} mode")
        super().train(mode)  # Call original method to update submodules
        # Add any custom logic here
        return self

    # -----------------------------
    # Override eval() method
    # -----------------------------
    def eval(self) -> Self:
        print("CustomBlock set to eval mode")
        return super().eval()


# Example usage
model = CustomBlock()
x = torch.randn(2, 4)

# Switch to training mode
model.train()
output_train = model(x)

# Switch to evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)

print("Output training mode:", output_train)
print("Output eval mode:", output_eval)


CustomBlock set to train mode
CustomBlock set to eval mode
CustomBlock set to eval mode
Output training mode: tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.]], grad_fn=<MulBackward0>)
Output eval mode: tensor([[0.0000, 0.2881, 0.7372, 0.0000],
        [0.0000, 0.2235, 0.3771, 0.0000]])


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:

---

# Часто используемые агрегаторы модулей в PyTorch

При построении нейронных сетей часто полезно объединять несколько слоёв или подмодулей в одну структуру.  
PyTorch предоставляет несколько **агрегаторов модулей**, которые помогают организовать слои и блоки. Наиболее распространённые:


## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.

---

## `nn.Sequential`

- Хранит модули в последовательном порядке.
- Выполняет их **в том порядке, в котором они добавлены** во время `forward`.
- Подходит для простых **последовательно соединённых слоёв** с одним входом и одним выходом.

**Ключевые моменты:**

- Прямой проход определяется автоматически.
- Не подходит для нескольких входов и ветвящейся логики.

In [13]:
seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)


nn.Sequential output shape: torch.Size([5, 2])


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.

---

## `nn.ModuleList`

- Хранит **список модулей**.
- **Не определяет `forward` автоматически**.
- Полезен, когда нужно **итерироваться по модулям** или использовать условные вычисления.

**Ключевые моменты:**

- Модули корректно регистрируются, поэтому параметры отслеживаются.
- Метод `forward()` нужно определить самостоятельно.

In [14]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

nn.ModuleList output shape: torch.Size([5, 2])


## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.

---

## `nn.ModuleDict`

- Хранит модули в **словаре** со строковыми ключами.
- Полезен для архитектур с **именованными ветвями**, **динамическим выбором** или **многоголовыми выходами**.
- Как и `ModuleList`, **не определяет `forward` автоматически**.

In [15]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({
            "branch1": nn.Linear(4, 8),
            "branch2": nn.Linear(4, 8)
        })
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)

md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)

nn.ModuleDict output shape: torch.Size([5, 2])



## Работа на семинаре

### LSTM

![lstm](assets/LSTM.png)

### Inception

![inception](assets/inception.png)

### Инсепшн (Inception)

### SE

![se](assets/SqueezeAndExcite.png)

### Selective Kernel

![selective](assets/SelectiveKernel.png)

### Селективное ядро (Selective Kernel)


### PatchMerger

![patchmerger](assets/PatchMerger.png)

### Слияние патчей (PatchMerger)


## Homework

2 задания:
1. Реализуйте требуемый в заголовке блок (максмсум 0.8 балов).

---

## Домашнее задание

## ResNet Block (0.1 балл)

![Resnet](assets/ResBlock.png)

https://arxiv.org/pdf/1512.03385


In [16]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.block(x)
        skip = self.shortcut(x)
        out = out + skip
        out = self.relu(out)
        return out

## Depthwise Separable Convolution (0.1 балл)
![DepthWiseConv](assets/DepthWiseConv.png)

https://arxiv.org/pdf/1610.02357

In [17]:
class SeparableConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, stride: int = 1, padding: int = 1, bias: bool = False):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels=in_channels,
            out_channels=in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            groups=in_channels,
            bias=bias
        )
        
        self.pointwise = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=1,
            bias=bias,
        )


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$

---

## Vanilla Attention (на русском)

Пусть:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

### Оценки выравнивания (Alignment Scores)

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

### Веса внимания (Attention Weights)

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

### Вектор контекста (Context Vector)

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

### Выход (Output)

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$

https://arxiv.org/abs/1409.0473

https://arxiv.org/abs/1508.04025

In [19]:
from typing import Optional
import torch
from torch import nn
import numpy as np

class VanillaAttention(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.w_align = nn.Linear(d, d, bias=False)
        self.w_value = nn.Linear(d, d, bias=True)
        self.w_query = nn.Linear(d, d, bias=True)


    def forward(self, query: torch.Tensor, key: torch.Tensor):
        q_align = self.w_align(query)
        score = torch.bmm(key, q_align.unsqueeze(-1)).squeeze(-1)
        att = torch.softmax(score, dim=1)
        context = torch.bmm(att.unsqueeze(1), key).squeeze(1)
        out = torch.tanh(self.w_value(context) + self.w_query(query))
        return out, att


## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$

---

## Скалярное внимание (Dot Product Attention) (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$

https://arxiv.org/abs/1706.03762


In [20]:
import math

class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout: float=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)


    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor):
        dk = q.size(-1)
        s = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(dk)
        attn = torch.softmax(s, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v)
        return out, attn


## Multihead Attention (0.1 балл)

![MultiheadAttention](assets/MultiheadAttention.webp)

https://arxiv.org/abs/1706.03762

In [21]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor):
        B, L, _ = q.shape

        q = self.w_q(q)
        k = self.w_k(k)
        v = self.w_v(v)

        q = q.view(B, L, self.num_heads, self.d_k).transpose(1, 2)
        k = k.view(B, L, self.num_heads, self.d_k).transpose(1, 2)
        v = v.view(B, L, self.num_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn, v)

        context = context.transpose(1, 2).contiguous().view(B, L, self.d_model)
        out = self.w_o(context)

        return out

## Transformer Encoder Layer (0.1 балл)


![Transformer Encoder Layer](assets/TransformerEncoder.png)


https://arxiv.org/abs/1706.03762

In [22]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)

        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)

        return x

## MLP Mixer (0.1 балл)


![MLPMixer](assets/MLPMixer.png)


https://arxiv.org/abs/2105.01601

In [23]:
class MLPMixerBlock(nn.Module):
    def __init__(self, num_patches: int, hidden_dim: int, token_mlp_dim: int, channel_mlp_dim: int):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.token_mlp = nn.Sequential(
            nn.Linear(num_patches, token_mlp_dim),
            nn.GELU(),
            nn.Linear(token_mlp_dim, num_patches),
        )

        self.norm2 = nn.LayerNorm(hidden_dim)
        self.channel_mlp = nn.Sequential(
            nn.Linear(hidden_dim, channel_mlp_dim),
            nn.GELU(),
            nn.Linear(channel_mlp_dim, hidden_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.norm1(x)
        y = y.transpose(1, 2)
        y = self.token_mlp(y)
        y = y.transpose(1, 2)
        x = x + y

        y = self.norm2(x)
        y = self.channel_mlp(y)
        x = x + y
        
        return x

## ConvMixer (0.1 балл)

![ConvMixer](assets/ConvMixer.png)


https://arxiv.org/abs/2201.09792

In [24]:
class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x):
        return x + self.fn(x)


class ConvMixer(nn.Module):

    def __init__(self, in_channels, dim, kernel_size, patch_size, depth, num_classes):
        super().__init__()

        self.patch_embed = nn.Sequential(
            nn.Conv2d(in_channels, dim, kernel_size=patch_size, stride=patch_size),
            nn.GELU(),
            nn.BatchNorm2d(dim),
        )

        blocks = []
        for _ in range(depth):
            blocks.append(
                nn.Sequential(
                    Residual(nn.Sequential(
                        nn.Conv2d(dim, dim, kernel_size=kernel_size, padding=kernel_size // 2, groups=dim),
                        nn.GELU(),
                        nn.BatchNorm2d(dim),
                    )),
                    nn.Conv2d(dim, dim, kernel_size=1),
                    nn.GELU(),
                    nn.BatchNorm2d(dim),
                )
            )
        self.blocks = nn.Sequential(*blocks)

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(dim, num_classes),
        )

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.blocks(x)
        x = self.head(x)
        return x

## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

---

Ответ: работают близко к Multihead Attention, потому что все они делают одно и то же на уровне идеи: смешивают информацию между токенами и информацию между каналами

Y=X+T(X),Z=Y+C(Y)

Multihead Attention
+: лучше захватывает дальние зависимости, адаптивные связи между любыми токенами
-: высокая стоимость по памяти/времени, особенно на длинных последовательностях

MLPMixer
+: простая архитектура, быстро и хорошо параллелится, часто легче в реализации
-: token-mixing менее адаптивен, чем attention, обычно чувствителен к объёму данных

ConvMixer
+: очень эффективен на изображениях, меньше вычислений, сильный локальный inductive bias
-: хуже моделирует глобальные зависимости без большой глубины/большого receptive field